**Cria um Volume para simular o landing dos arquivos do ERP / Exporta amostras de dados de todas as tabelas**

In [0]:
# Cria o catalog/schema/volume
spark.sql("CREATE CATALOG IF NOT EXISTS erp_lakehouse")
spark.sql("CREATE SCHEMA IF NOT EXISTS erp_lakehouse.landing")
spark.sql("CREATE VOLUME IF NOT EXISTS erp_lakehouse.landing.raw_files")
spark.sql("CREATE VOLUME IF NOT EXISTS erp_lakehouse.landing.pipeline_metadata")


# Confirma o caminho físico do volume

volume_path = "/Volumes/erp_lakehouse/landing/raw_files"

tabelas = ["customer", "orders", "lineitem", "part", "supplier"]


for tabela in tabelas:
    df = spark.table(f"samples.tpch.{tabela}")
    (df.limit(50000)  # 50.000 linhas de amostra de cada tabela
       .write
       .mode("overwrite")
       .format("csv")
       .option("header", "true")
       .save(f"{volume_path}/{tabela}"))
    print(f"{tabela}: {df.limit(50000).count()} linhas exportadas")


In [0]:
orders_full = spark.table("samples.tpch.orders")
lineitem_full = spark.table("samples.tpch.lineitem")
customer_full = spark.table("samples.tpch.customer")
part_full = spark.table("samples.tpch.part")
supplier_full = spark.table("samples.tpch.supplier")

# 1. Tabela âncora: uma amostra de pedidos
orders_sample = orders_full.limit(50000)

# 2. Lineitem: só os itens que pertencem a esses pedidos (join real preserva a FK)
lineitem_sample = (lineitem_full
    .join(orders_sample.select("o_orderkey"),
          lineitem_full.l_orderkey == orders_sample.o_orderkey, "inner")
    .select(lineitem_full["*"]))

# 3. Customer: só os clientes que fizeram esses pedidos
customer_sample = (customer_full
    .join(orders_sample.select("o_custkey").distinct(),
          customer_full.c_custkey == orders_sample.o_custkey, "inner")
    .select(customer_full["*"]))

# 4. Part: só os produtos que aparecem nesses itens
part_sample = (part_full
    .join(lineitem_sample.select("l_partkey").distinct(),
          part_full.p_partkey == lineitem_sample.l_partkey, "inner")
    .select(part_full["*"]))

# 5. Supplier: só os fornecedores que aparecem nesses itens
supplier_sample = (supplier_full
    .join(lineitem_sample.select("l_suppkey").distinct(),
          supplier_full.s_suppkey == lineitem_sample.l_suppkey, "inner")
    .select(supplier_full["*"]))

samples = {
    "customer": customer_sample,
    "orders": orders_sample,
    "lineitem": lineitem_sample,
    "part": part_sample,
    "supplier": supplier_sample,
}

In [0]:
# Limpa landing zone e metadados do pipeline
dbutils.fs.rm("/Volumes/erp_lakehouse/landing/raw_files", recurse=True)
dbutils.fs.rm("/Volumes/erp_lakehouse/landing/pipeline_metadata/_checkpoints", recurse=True)
dbutils.fs.rm("/Volumes/erp_lakehouse/landing/pipeline_metadata/_schemas", recurse=True)

# Dropa as tabelas de todas as camadas para recomeçar limpo
for camada in ["bronze", "silver"]:
    for tabela in ["customer", "orders", "lineitem", "part", "supplier"]:
        spark.sql(f"DROP TABLE IF EXISTS erp_lakehouse.{camada}.{tabela}")

spark.sql("DROP TABLE IF EXISTS erp_lakehouse.gold.fact_sales")
spark.sql("DROP TABLE IF EXISTS erp_lakehouse.gold.sales_summary_monthly")

print("✅ Ambiente limpo, pronto para reprocessar")

In [0]:
volume_path = "/Volumes/erp_lakehouse/landing/raw_files"

for tabela, df in samples.items():
    (df.write
       .mode("overwrite")
       .format("csv")
       .option("header", "true")
       .save(f"{volume_path}/{tabela}"))
    print(f"{tabela}: {df.count()} linhas exportadas")